# EM Classification — feature service 기반 테스트 (torch-free)

백본은 **서버(feature_service)**가 들고 있고, 이 노트북은 embedding 만 받아 분류한다.
→ **torch 불필요**, numpy + PIL + (requests/urllib) 만 필요.

> ⚠️ **feature_service 가 켜져 있어야 동작.** 서버 기동:
> `EM_CKPT=/path/to/teacher.pth bash scripts/serve_features.sh`
>
> 백본을 in-process 로 직접 로드하는 버전은 `em_classification_demo.ipynb` 참고
> (그건 service 불필요, torch+GPU 필요).

## 0. 경로/임포트 (torch 없음)

In [ ]:
import os, sys, json, urllib.request
from pathlib import Path

# repo 루트를 path 에 (inspection import 용)
_HERE = Path.cwd(); _REPO = _HERE
while not (_REPO / 'inspection').exists() and _REPO != _REPO.parent:
    _REPO = _REPO.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
print('repo:', _REPO)

import numpy as np
from PIL import Image
from inspection.service.client_example import get_features
from inspection.em_classifier import ClassifierHead, DefectRegistry
print('torch-free client OK')

## 1. 설정
- `SERVER`: feature_service 주소
- `DATA_ROOT`: 클래스별 폴더(학습)  ·  `TEST_DIR`: 분류할 이미지 폴더

In [ ]:
SERVER    = 'http://localhost:8000'   # feature_service 주소
DATA_ROOT = '/path/to/class_folders'  # FILL_IN (classA/ classB/ ...)
TEST_DIR  = '/path/to/test_images'    # FILL_IN (라벨 없어도 됨)
OUT       = './out/service_head.npz'
CLF       = 'logreg'                  # logreg | ncm | tip
os.makedirs(Path(OUT).parent, exist_ok=True)
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

## 2. 서비스 상태 확인 (`GET /health`)

In [ ]:
with urllib.request.urlopen(SERVER.rstrip('/') + '/health', timeout=30) as r:
    info = json.loads(r.read().decode())
model_info = info.get('model', {})
print('status:', info.get('status'), '| model:', model_info)
assert info.get('status') == 'ok', 'feature_service 가 정상 기동됐는지 확인 (serve_features.sh)'

## 3. 학습 — 폴더별 embedding 수집 → registry/head
각 클래스 폴더 이미지를 서버 `/features` 로 보내 embedding 을 받아 `DefectRegistry` 구성.
백본 forward 는 전부 서버에서 일어난다.

In [ ]:
root = Path(DATA_ROOT)
class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
assert class_dirs, f'클래스 폴더 없음: {root}'

reg = DefectRegistry(meta={'server': SERVER, **model_info})
for d in class_dirs:
    paths = [str(p) for p in sorted(d.rglob('*')) if p.suffix.lower() in IMG_EXTS]
    if not paths:
        print('  (건너뜀, 비어있음)', d.name); continue
    feats, _ = get_features(SERVER, paths, feature_kind=model_info.get('feature_kind'))
    reg.enroll(d.name, feats)
    print(f'  {d.name}: {len(paths)}장')

head = reg.build_head(kind=CLF)
head.save(OUT)
print('classes:', head.class_names, '| dim:', head.feature_dim, '| saved:', OUT)

## 4. 분류 — 테스트 폴더
저장한 head 를 로드해 `TEST_DIR` 이미지를 분류(역시 embedding 은 서버에서).

In [ ]:
head = ClassifierHead.load(OUT)
paths = [str(p) for p in sorted(Path(TEST_DIR).rglob('*')) if p.suffix.lower() in IMG_EXTS]
print(f'{len(paths)} images in {TEST_DIR}')
feats, names = get_features(SERVER, paths, feature_kind=head.meta.get('feature_kind'))
proba = head.predict_proba(feats)
k = min(3, len(head.class_names))
rows = []
for name, p in zip(names, proba):
    order = p.argsort()[::-1][:k]
    tops = ', '.join(f'{head.class_names[j]}:{p[j]:.2f}' for j in order)
    lab = head.class_names[int(order[0])]
    print(f'{name}\t-> {lab} ({p[order[0]]:.3f})\t[{tops}]')
    rows.append({'file': name, 'pred': lab, 'score': float(p[order[0]])})

In [ ]:
# 결과 표/분포 (pandas 있으면)
try:
    import pandas as pd
    df = pd.DataFrame(rows); display(df.head(20))
    print('클래스별 분포:'); print(df['pred'].value_counts())
except Exception:
    pass

## 5. 참고 — 두 실행 모드

| 노트북 | 백본 | service | torch |
|---|---|---|---|
| **이 노트북** (service) | 서버가 로드 | **필요** | 불필요(클라) |
| `em_classification_demo.ipynb` (direct) | in-process(`EMFeatureExtractor`) | 불필요 | 필요(+GPU) |

지금 대부분의 도구(em_classifier CLI, separability, heatmap)는 **direct** 모드다.
이 노트북은 **service** 모드 경로를 검증한다.